# 11. 상수 스윕 — Colab (범용)

챔피언 스택(`wonwoo` s24)의 **후처리 상수**를 바꿔 제출 zip을 찍는다.
노트북 10은 어제 실험의 기록이므로 건드리지 않고, 이쪽을 반복해서 쓴다.

## 10과 무엇이 다른가

| | 10 | 11 |
| --- | --- | --- |
| 모델 | 전체 학습 30분 | **`p_a40.zip`에서 복구, 학습 없음** |
| 스윕 대상 | `MLP_ALPHA` 고정 | **임의의 config 상수** |
| 소요 | 약 1시간 | **약 15분** |

## 스윕 가능한 상수 (재학습 불필요)

`train.py`가 `--artifacts-only`에서도 **config에서 다시 읽어 재계산**하는 것들이다.

| 상수 | 배포값 | 성격 |
| --- | --- | --- |
| `MLP_ALPHA` | 0.40 → **0.60 (신챔피언)** | lgbm/mlp 혼합비 |
| `CTX_AXES` β | hand 1.875, count4 1.25 | δ 보정 강도 |
| `CTX_K` | 200 | δ 표 수축 |
| `CTX_HALF_LIFE` | 1.0 | δ 표 recency |
| `ASOF_AXES` β | 0.03, 0.03 | 시즌 진행분 강도 |
| `ASOF_STRENGTH` | 35 | 시즌 진행분 수축 |

## 스윕할 수 없는 것 (조용히 무시된다)

- **`MLP_VARIANT_WEIGHT`** — `train.py:380`이 `old["mlp_weights"]`를 읽는다.
  config를 바꿔도 `--artifacts-only`에서는 **아무 일도 일어나지 않는다.** 전체 재학습이 필요하다.
- `SMOOTH_K_*`, `FINAL_LGB_PARAMS`, `MLP_VARIANTS` 등 피처·모델을 바꾸는 것 —
  피처 해시 가드에 걸려 실행이 중단된다 (조용한 오류는 아니다).

## 규정상 열지 않는 것

**`TARGET_PRIOR_2025`는 LB로 스윕하지 않는다.** 레벨 손실은 예측평균 = 실제평균일 때
최소가 되므로, LB 곡선의 꼭짓점은 곧 **평가 데이터 라벨의 평균값**이다. 그것을 측정해
전 행의 예측을 이동시키는 것은 규칙이 금지하는 "전체 평가 데이터의 분포를 이용한 보정"에
해당한다 (원우 규칙 2). α 스윕과는 성질이 다르다 — α는 두 예측기 중 무엇이 잘
일반화하는지를 고르는 모델 선택이다.

## 준비물

```
MyDrive/LG_aimers_9th/
├── data/                          train.csv 등 (이미 있음)
├── champion_src.tar.gz            (이미 있음)
└── outputs/alpha_probe/p_a40.zip  모델 복구원 (이미 있음)
```

런타임: **CPU · 고용량 RAM** (8 vCPU). GPU 아님 — torch 휠이 달라진다.

## 1. 환경

In [ ]:
import sys, platform, subprocess
print('Python  ', sys.version.split()[0])
print('Platform', platform.machine(), platform.system(), '· vCPU', __import__('os').cpu_count())
assert platform.machine() == 'x86_64', 'x86_64가 아니다 — 재현 충실도가 떨어진다'

PINS = ['lightgbm==4.6.0', 'numpy==2.2.6', 'pandas==2.3.3', 'scipy==1.15.2',
        'joblib==1.4.2', 'scikit-learn==1.6.1', 'pyarrow==19.0.1']
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', *PINS], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch==2.13.0+cpu',
                '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
print('설치 완료')

In [ ]:
# 불일치가 있으면 [런타임] -> [세션 다시 시작] 후 이 셀부터 다시 실행
import importlib.metadata as md
EXPECT = {'lightgbm':'4.6.0','numpy':'2.2.6','pandas':'2.3.3','scipy':'1.15.2',
          'joblib':'1.4.2','scikit-learn':'1.6.1','pyarrow':'19.0.1'}
bad = [(p, w, md.version(p)) for p, w in EXPECT.items() if md.version(p) != w]
for p, w in EXPECT.items():
    print(f"{'ok ' if md.version(p)==w else 'X  '}{p:16s} {md.version(p):12s} (기대 {w})")
print(f"{'ok ' if md.version('torch').startswith('2.13.0') else 'X  '}{'torch':16s} {md.version('torch')}")
assert not bad, f'버전 불일치: {bad} — 런타임 재시작 후 다시 실행'

## 2. 데이터 · 코드 · 모델 복구

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, shutil, os, tarfile, zipfile, json, time, textwrap
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/LG_aimers_9th')
ROOT  = Path('/content/champion')
(ROOT / 'data').mkdir(parents=True, exist_ok=True)

with tarfile.open(DRIVE / 'champion_src.tar.gz') as tf:
    tf.extractall(ROOT)
assert (ROOT/'src'/'train.py').is_file() and (ROOT/'submit'/'script.py').is_file()
print('코드 전개:', len(list((ROOT/'src').glob('*.py'))), '개 .py')

for name in ['train.csv', 'test.csv', 'sample_submission.csv']:
    src, dst = DRIVE/'data'/name, ROOT/'data'/name
    assert src.is_file(), f'{src} 없음'
    if not dst.exists():
        os.symlink(src, dst)

h = hashlib.sha256()
with open(ROOT/'data'/'train.csv', 'rb') as f:
    for blk in iter(lambda: f.read(1 << 20), b''):
        h.update(blk)
assert h.hexdigest() == 'd2081186b458b49f60b082be480c273135833e15ba59a76d033af28bcf8763ff', \
    'train.csv 가 기록된 파일과 다르다'
print('train.csv 무결성 확인')

In [ ]:
# 모델 복구 — p_a40.zip 의 model/ 이 곧 어제의 models/a40 이다 (파일 28개).
# `--artifacts-only` 가 필요로 하는 것은 이 28개 + train.csv 뿐이라 재학습이 없다.
BASE = ROOT / 'models' / 'a40'
SRC_ZIP = DRIVE / 'outputs' / 'alpha_probe' / 'p_a40.zip'
assert SRC_ZIP.is_file(), f'{SRC_ZIP} 없음'

BASE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SRC_ZIP) as z:
    for n in z.namelist():
        if n.startswith('model/') and not n.endswith('/'):
            (BASE / Path(n).name).write_bytes(z.read(n))

n_l = len(list(BASE.glob('lgbm_*.txt')))
n_m = len(list(BASE.glob('mlp_*.npz')))
assert n_l == 20 and n_m == 8, f'lgbm {n_l} · mlp {n_m} (기대 20 · 8)'
assert (BASE/'artifacts.joblib').is_file() and (BASE/'meta.json').is_file()
meta = json.loads((BASE/'meta.json').read_text())
print(f'복구 완료: {BASE}')
print(f'  lgbm {n_l} · mlp {n_m} · 복구원 mlp_alpha={meta["mlp_alpha"]}')

## 3. 스윕 정의

`SWEEP`의 각 항목이 zip 하나가 된다. **이 셀만 바꾸면 된다** — 아래 셀들은 전부 `SWEEP`을
순회할 뿐이고 태그를 하드코딩한 곳이 없다. (`models/a40`은 스윕 대상이 아니라 **모델 복구원**이다.)

### 2026-08-21 2차 — MLP 변형 비중 (`noid` 몫)

α 축은 8점으로 종결됐다 (`α* = 0.74499`, 신기록 **1076.0280976273**).
다음은 **MLP 내부의 변형 비중**이다. α와 같은 성질의 질문 — 예측기 간 상대 신뢰도다.

**왜 이 축인가.** 배포값은 `emb4 0.625 : noid 0.375`인데 `wm = vw/tot/nseed`로 펼치면
`emb4 0.625/5 = 0.125`, `noid 0.375/3 = 0.125`로 **8멤버 균등 평균**이다.
그런데 §7.24가 이렇게 기록했다:

> `noid`는 핸디캡 멤버가 아니라 **더 센** 멤버였다.

**더 센 멤버가 3/8만 갖고 있다.** §7.27이 비율 변경을 기각했지만 그건 홀드아웃 게이트였고,
α에서 그 게이트가 부호를 틀린 것이 증명됐다.

| `noid` 몫 | 태그 | 멤버당 가중 (emb4 / noid) | LB |
| ---: | --- | --- | ---: |
| **0.375** | (`p_a75`) | 0.1250 / 0.1250 | **1076.0280976273** |
| 0.500 | `n50` | 0.1000 / 0.1667 | 측정 대상 |
| 0.625 | `n62` | 0.0750 / 0.2083 | 측정 대상 |

챔피언이 그대로 기준점이라 **2슬롯으로 3점 곡선**이 나온다.

### 이 축은 config 로 못 바꾼다 — 아티팩트를 직접 고친다

`train.py:380`이 `old["mlp_weights"]`를 읽으므로 `config.MLP_VARIANT_WEIGHT`를 바꿔도
`--artifacts-only`에서는 **조용히 무시된다.** 대신 `artifacts.joblib`의 `mlp_weights`를
직접 패치한다. 이 값은 순수한 선형 가중 벡터이고 세 곳이 같은 것을 읽으므로 일관된다:

```
load_members(MODEL_DIR, old, ...)   ->  z_mlp_ref 가 새 가중으로 계산되고
raw_ref = (1-α)·lgbm + α·z_mlp      ->  delta 가 그 위에서 재해결되고
_finish(..., mlp_wts, ...)          ->  같은 가중이 zip 에 저장된다
```

`MLP_ALPHA`는 두 태그 모두 **0.75 고정** — 한 번에 한 축만 움직인다.

In [ ]:
# 태그 -> 오버라이드. 라운드마다 이 dict 만 바꾸고 아래 셀들을 다시 실행한다.
#
# 일반 키       : config 속성명 (train import 전에 setattr)
# '_MLP_VW' 키  : MLP 변형 비중. config 로는 안 먹으므로 artifacts.joblib 을 패치한다.
#
# 2026-08-21 2차 — noid 몫. 기준점은 p_a75 = 1076.0280976273 (noid 0.375, 8멤버 균등).
SWEEP = {
    'n50': {'MLP_ALPHA': 0.75, '_MLP_VW': {'emb4': 0.500, 'noid': 0.500}},
    'n62': {'MLP_ALPHA': 0.75, '_MLP_VW': {'emb4': 0.375, 'noid': 0.625}},
}

# ── 지난 라운드 ───────────────────────────────────────────────────
# 08-20 1차  a20..a60   MLP_ALPHA 0.20~0.60   -> 1074.5321158142
# 08-21 1차  a70,a75,a80                      -> 1076.0280976273, α 축 종결
#
# ── 남은 후보 (전부 --artifacts-only) ─────────────────────────────
#   후처리 4축 공통 배율  {'MLP_ALPHA':0.75, 'CTX_AXES':[['hand',1.5],['count4',1.0]],
#                          'ASOF_AXES':[['gap_success',0.024],['gap_bsuccess',0.024]]}
#   CTX_K          200   {'MLP_ALPHA':0.75, 'CTX_K': 100}  /  400
#   CTX_HALF_LIFE  1.0   {'MLP_ALPHA':0.75, 'CTX_HALF_LIFE': 0.5}  /  2.0
#   ASOF_STRENGTH  35    {'MLP_ALPHA':0.75, 'ASOF_STRENGTH': 20}  /  60
#
# TARGET_PRIOR_2025 는 열지 않는다 (규정 — 노트북 상단).
#   2026-08-21 학습 데이터로 재추정을 시도했으나 시즌말 기준 예측기가 시즌평균 기준보다
#   나빴다 (MAE 0.0155 vs 0.0120). 배포값 0.475 가 최선으로 확인돼 축을 닫았다.

for tag, ov in SWEEP.items():
    assert len(f'p_{tag}.zip') <= 30, f'{tag}: 파일명 30자 초과'
    print(f'p_{tag}.zip')
    for k, v in ov.items():
        print(f'    {k:12s} {v}')
    vw = ov.get('_MLP_VW')
    if vw:
        assert abs(sum(vw.values()) - 1.0) < 1e-9, f'{tag}: _MLP_VW 합이 1이 아니다'
        print(f'    {"":12s} -> 멤버당  emb4 {vw["emb4"]/5:.4f}  noid {vw["noid"]/3:.4f}')

In [ ]:
RUNNER = ROOT / 'run_sweep.py'
RUNNER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', required=True)
    ap.add_argument('--reuse', required=True)
    ap.add_argument('--overrides', required=True)   # JSON (config 속성)
    ap.add_argument('--mlp-vw', default=None)       # JSON (변형 -> 비중), 아티팩트 패치
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    out = ROOT / a.out
    if out.exists(): shutil.rmtree(out)
    shutil.copytree(ROOT / a.reuse, out)      # 복구원을 보존한다
    os.environ['LGA_MODEL_DIR'] = str(out)
    sys.path.insert(0, str(ROOT / 'src'))
    os.chdir(ROOT)

    # ── MLP 변형 비중은 config 로 못 바꾼다 (train.py:380 이 old["mlp_weights"] 를
    #    읽는다). artifacts.joblib 을 직접 고친다. 순수 선형 가중 벡터라 안전하고,
    #    load_members / _finish / predict.py 가 모두 같은 값을 읽어 일관된다.
    if a.mlp_vw:
        import joblib
        vw = json.loads(a.mlp_vw)
        b = joblib.load(out / 'artifacts.joblib')
        tags, cur = b['mlp_members'], list(b['mlp_weights'])

        # 배포 상태가 8멤버 균등인지 먼저 확인한다 — 우리 기전 이해의 검증이다.
        assert max(cur) - min(cur) < 1e-9, f'배포 가중이 균등이 아니다: {cur}'
        assert abs(sum(cur) - 1.0) < 1e-9, f'가중 합이 1이 아니다: {sum(cur)}'

        variant = [t.rsplit('_', 1)[0] for t in tags]       # 'emb4_42' -> 'emb4'
        assert set(variant) == set(vw), f'변형 불일치 {set(variant)} vs {set(vw)}'
        cnt = {v: variant.count(v) for v in vw}
        new = [vw[v] / sum(vw.values()) / cnt[v] for v in variant]
        assert abs(sum(new) - 1.0) < 1e-9, f'새 가중 합이 1이 아니다: {sum(new)}'

        print('mlp_weights 패치:', flush=True)
        for t, o, n in zip(tags, cur, new):
            print(f'    {t:18s} {o:.6f} -> {n:.6f}', flush=True)
        b['mlp_weights'] = new
        joblib.dump(b, out / 'artifacts.joblib', compress=3)

    # train 을 import 하기 **전에** config 를 고친다.
    # train.py 는 `from config import (...)` = import 시점의 값 복사이므로,
    # 순서가 뒤바뀌면 조용히 배포값으로 돌아간다.
    import config
    ov = json.loads(a.overrides)
    for k, v in ov.items():
        assert hasattr(config, k), f'config 에 {k} 가 없다'
        if isinstance(v, list):               # [[name, beta], ...] -> [(name, beta), ...]
            v = [tuple(x) if isinstance(x, list) else x for x in v]
        print(f'주입 {k}: {getattr(config, k)} -> {v}', flush=True)
        setattr(config, k, v)

    sys.argv = ['train.py', '--artifacts-only']
    import train
    for k, v in ov.items():                   # 침묵 실패 방어선
        if hasattr(train, k):
            got = getattr(train, k)
            exp = [tuple(x) if isinstance(x, list) else x for x in v] if isinstance(v, list) else v
            assert got == exp, f'주입 실패 train.{k}={got} != {exp}'
    assert 'mlp_alpha' not in train._TRAIN_SHAPING, (
        'alpha 가 모델 결정 설정에 들어갔다 — artifacts-only 재사용이 위험하다')
    train.main()

    if a.mlp_vw:                              # 저장된 zip 이 패치본을 담았는지 확인
        import joblib
        chk = joblib.load(out / 'artifacts.joblib')['mlp_weights']
        assert max(chk) - min(chk) > 1e-9, '저장 후 가중이 다시 균등이다 — 패치가 덮였다'
        print(f'저장 확인: mlp_weights {[round(x,6) for x in chk]}', flush=True)
'''), encoding='utf-8')


def run(tag, ov):
    """출력을 한 줄씩 즉시 흘린다. capture_output 은 진행 상황을 가린다."""
    ov = dict(ov)
    vw = ov.pop('_MLP_VW', None)
    cmd = [sys.executable, '-u', str(RUNNER), '--out', f'models/{tag}',
           '--reuse', 'models/a40', '--overrides', json.dumps(ov)]
    if vw:
        cmd += ['--mlp-vw', json.dumps(vw)]
    p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if p.wait():
        raise SystemExit(f'{tag} 실패 (종료코드 {p.returncode})')
    return ''.join(lines)


print('러너 작성 완료 — _MLP_VW 는 artifacts.joblib 패치 경로로 간다')

## 4. 스윕 실행 — 태그당 약 45초

`--artifacts-only`이므로 모델은 재사용하고 δ 표 재적합 + prior shift 재해결만 한다.
`logit_delta`가 태그마다 **다른 값**이어야 한다 — 같으면 주입이 안 먹은 것이다.

In [ ]:
import re
deltas = {}
for tag, ov in SWEEP.items():
    print(f'\n{"="*22} {tag} {"="*22}')
    log = run(tag, ov)
    m = re.search(r'logit_delta=([-+0-9.]+)', log)
    deltas[tag] = float(m.group(1)) if m else None
    for probe in ['셀 1,571개', '셀 3,105개', 'features=78']:
        assert probe in log, f'{tag}: 기대 로그 누락 — {probe}'

print('\n' + '='*46)
for tag, d in deltas.items():
    print(f'  {tag:6s} logit_delta = {d}')
assert len(set(deltas.values())) == len(deltas), \
    'logit_delta 가 겹친다 — 오버라이드가 안 먹었을 가능성이 크다'
print('  전부 다른 값 — 주입이 반영됐다')

## 5. zip 빌드

원우 `build_submit.py`를 감싼다. 그대로 쓰면 네 가지가 깨진다:
`model_dir` 하드코딩 · `submit/` 전체를 훑는 zip · 원우 추적 파일 기록 · s25.. 자동 번호.

In [ ]:
BUILDER = ROOT / 'build_one.py'
BUILDER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, zipfile, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', required=True)
    ap.add_argument('--name', required=True)
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    zip_path = ROOT / f'{a.name}.zip'
    assert len(zip_path.name) <= 30, f'파일명 {len(zip_path.name)}자 > 30자'
    if zip_path.exists(): zip_path.unlink()

    staged = ROOT / 'submit' / 'model'
    if staged.exists(): shutil.rmtree(staged)
    shutil.copytree(ROOT / a.models, staged)
    alpha = json.loads((staged/'meta.json').read_text())['mlp_alpha']

    strays = [p for p in (ROOT/'submit').iterdir()
              if p.is_dir() and p.name.startswith('model') and p.name != 'model']
    assert not strays, f'submit/ 에 다른 모델이 있다 (zip 이 통째로 삼킨다): {strays}'

    sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
    import build_submit
    build_submit.record_history = lambda *a, **k: print('  이력 기록 건너뜀')
    build_submit.log_submission = lambda *a, **k: print('  SUBMISSIONS.md 기록 건너뜀')
    build_submit.next_zip_path  = lambda meta, suffix='': zip_path
    build_submit.main()

    with zipfile.ZipFile(zip_path) as z: names = z.namelist()
    n_l = sum(n.startswith('model/lgbm_') for n in names)
    n_m = sum(n.startswith('model/mlp_')  for n in names)
    assert n_l == 20 and n_m == 8, f'LGBM {n_l} · MLP {n_m} (기대 20 · 8)'
    print(f'OK {zip_path.name} alpha={alpha} lgbm={n_l} mlp={n_m} '
          f'{zip_path.stat().st_size/1e6:.1f}MB')
'''), encoding='utf-8')


def build(tag):
    p = subprocess.Popen([sys.executable, '-u', str(BUILDER),
                          '--models', f'models/{tag}', '--name', f'p_{tag}'],
                         cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise SystemExit(f'{tag} 빌드 실패 (종료코드 {p.returncode})')


for tag in SWEEP:
    print(f'\n{"="*22} {tag} {"="*22}', flush=True)
    build(tag)

## 6. 검증 — 압축을 푼 상태에서 실행

`build_submit.py`는 `script.py`를 `submit/` 작업 디렉터리에서 돌리므로
**`submit/`에는 있고 zip에는 없는 파일**을 못 잡는다. 여기서는 빈 임시 디렉터리에 푼다.

행 독립성 임계값 `TOL = 1e-6` — 어제 측정한 잔차는 MLP BLAS 노이즈이며 α에 정비례했다
(2.1e-08/α). 진짜 누출은 1e-3 이상이라 두 자릿수 이상 떨어져 있다.

In [ ]:
import tempfile, pandas as pd
sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
import build_submit as bs

N_ROWS, TOL = 20000, 1e-6
dummy = bs.make_dummy_test(N_ROWS)
HIDDEN = ('__MACOSX', '.DS_Store', 'Thumbs.db', '.ipynb_checkpoints')

def run_script(work, df):
    d = work/'data'; d.mkdir(exist_ok=True)
    df.to_csv(d/'test.csv', index=False)
    pd.DataFrame({'row_id': df['row_id'], 'control_success': .5}).to_csv(
        d/'sample_submission.csv', index=False)
    shutil.rmtree(work/'output', ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, 'script.py'], cwd=work,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:]); print(r.stderr[-3000:]); raise SystemExit('script.py 실패')
    return pd.read_csv(work/'output'/'submission.csv'), time.time()-t0

for tag in SWEEP:
    zp = ROOT/f'p_{tag}.zip'
    print(f'\n{"="*22} {zp.name} {"="*22}')
    assert len(zp.name) <= 30
    with zipfile.ZipFile(zp) as z:
        assert z.testzip() is None, 'CRC 불일치'
        names = z.namelist()
    assert not [n for n in names if any(h in n for h in HIDDEN)], '숨은 파일'
    for req in ['script.py', 'requirements.txt']:
        assert req in names, f'루트에 {req} 없음'
    assert 'script.py' in {n.split('/', 1)[0] for n in names}, '최상위 래퍼 폴더가 있다'
    print(f'  구조 정상 · 구성원 {len(names)}개 · 파일명 {len(zp.name)}자')

    with tempfile.TemporaryDirectory() as tmp:
        work = Path(tmp)/'x'; work.mkdir()
        with zipfile.ZipFile(zp) as z: z.extractall(work)
        assert 'torch' not in (work/'requirements.txt').read_text(), 'requirements 에 torch'

        base, el = run_script(work, dummy)
        proj = el/len(dummy)*245789
        assert list(base.columns) == ['row_id','control_success']
        assert len(base) == len(dummy)
        assert base['row_id'].equals(dummy['row_id'].reset_index(drop=True)), 'row_id 불일치'
        p = base['control_success']
        assert p.notna().all() and p.between(0,1).all()
        assert proj < 480, f'추론 시간 여유 부족 ({proj:.0f}s)'
        print(f'  실행 {el:.1f}s -> 245,789행 환산 {proj:.0f}s / 600s ({proj/6:.0f}%)')
        print(f'  출력 정상  mean={p.mean():.6f}  min={p.min():.4f}  max={p.max():.4f}')

        ref = base.set_index('row_id')['control_success']
        sh, _ = run_script(work, dummy.sample(frac=1, random_state=42).reset_index(drop=True))
        d1 = (sh.set_index('row_id')['control_success'] - ref).abs().max()
        half = len(dummy)//2
        ch = pd.concat([run_script(work, dummy.iloc[:half].reset_index(drop=True))[0],
                        run_script(work, dummy.iloc[half:].reset_index(drop=True))[0]])
        d2 = (ch.set_index('row_id')['control_success'] - ref).abs().max()
        assert d1 < TOL, f'셔플 후 예측이 달라진다 ({d1:.3e}) — 규정 위반 의심'
        assert d2 < TOL, f'청크 분할 예측이 달라진다 ({d2:.3e}) — 규정 위반 의심'
        print(f'  행 독립성  셔플 {d1:.1e} · 청크 {d2:.1e}'
              + ('' if max(d1,d2) == 0 else '  (BLAS 노이즈)'))

print('\n' + '='*46)
print(f'{len(SWEEP)}개 전부 통과. 제출 가능.')
print('='*46)

## 7. Drive 저장

결과는 `hyejin:reports/10_alpha_probe_report.md` §5 표에 이어서 기록한다.

In [ ]:
OUT = DRIVE / 'outputs' / 'alpha_probe'
OUT.mkdir(parents=True, exist_ok=True)
for tag in SWEEP:
    src = ROOT / f'p_{tag}.zip'
    shutil.copy2(src, OUT / src.name)
    print(f'{OUT}/{src.name}   ({src.stat().st_size/1e6:.1f} MB)')

print('\n제출할 것:')
for i, t in enumerate(SWEEP, 1):
    print(f'  {i}. p_{t}.zip')
print('\n비교 기준: p_a75.zip = 1076.0280976273  (α=0.75, 보정축 배율 m=1.0)')
print('결과는 hyejin:reports/10_alpha_probe_report.md 에 기록한다.')